# 10 · M1.6 — M1.5 with production's joint design: a new Student-t shock per leg

## Executive summary (read this first)

**What production does for the joint (cross-asset) score** (`forecast_models._cumulative_walk_model`), and whether M1.5 has it:

| production's joint design | M1.5 | **M1.6** |
|---|---|---|
| 1. one accumulating path per draw: horizon 2 is reached by walking through horizon 1, so corr(h₁, h₂) = √(h₁ ÷ h₂) | yes (M0's `min(h₁, h₂) × Σ`) | yes |
| 2. inside each leg, a Cholesky shock correlated across assets; correlation from date-aligned steps | yes (the 260 steps, date-aligned) | yes |
| 3. multivariate Student-t: **a new χ² per draw for every leg**, shared by the leg's assets | **no**: one χ² per draw for the whole path | **yes** |

So the one piece of production's design that M1.5 lacks is **(3)**. **M1.6 = M1.5 + a new χ² for every leg.**

**What that changes, in one draw.** A Student-t draw is a normal draw × √((ν − 2) ÷ χ²). A small χ² makes the draw wild.
- In M1.5, one χ² scales the whole path. A wild draw is wild on leg 1 *and* on leg 2: the path is all calm or all wild.
- In M1.6, leg 1 and leg 2 each get their own χ². A wild first month can be followed by a calm second one.
- The sd of each leg stays the same (the factor has mean square 1), and so does every correlation. What changes is how the tails
  of the two horizons move together, which is what the variogram (joint) score sees.

Everything else is M1.5: transform per asset type, the drift rule (CPI and NFP only), the EWMA sd (halflife 63), the correlation
and ν from the last 260 steps, and the Student-t only on rates, FX and equity. Nothing is tuned.

**Same random numbers as M1.5.** M1.6 takes M1.5's normal path, cuts it into legs, rescales each leg by its own χ², and adds the
legs back up. Leg 1 uses the same χ² as M1.5. So the 33 one-horizon units draw exactly M1.5's numbers, and any difference
comes only from the later legs.

**The test:** notebook 01's backtest dates and answers; competition score per unit vs M0 (below 1 beats M0).

**Result: M1.6 does not beat M1.5.** All units 0.9448 vs 0.9444; F3 0.9151 vs 0.9141; better on only 14 of the 40 units
with 2+ horizons. The joint part barely moves (F3: 0.9079 vs 0.9073 × M0) and the tail part gets worse (0.932 vs 0.929).
**Keep M1.5's one χ² for the whole path.** Why: §6. Production's other joint pieces (1 and 2) are already in M1.5.

Results: §2 (every model), §3 (per family, asset type, and score part), §5 (in short), §6 (why). No realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: M1.5 with an optional new χ² per leg

`m13_draws(U, origin, ewma_halflife=63, t_per_leg=True)` is M1.6; `t_per_leg=False` is M1.5.

Checks:
1. `t_per_leg=False` reproduces notebook 09's M1.5.
2. A one-horizon unit draws exactly the same numbers under M1.5 and M1.6.
3. On a two-horizon unit, each leg's sd and the leg-to-leg correlation are unchanged; only the joint tails differ.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

def t_per_leg_scale(dev, cells, hs, t_cells, nu, rng_t):
    """Cut each draw's path into legs (horizon 1, horizon 1 -> 2, ...), give every leg its OWN chi-square, add the legs back up.

    dev is M1.5's normal path (cells sorted by asset, then horizon). Its legs are independent and, inside a leg, correlated across
    assets with cov = new steps x sigma -- exactly production's per-leg Cholesky draw. Leg 1 takes the same chi-square as M1.5,
    so a 1-horizon unit draws the same numbers as M1.5; only the later legs change."""
    k = len(hs)
    if k == 1:
        return dev * np.where(t_cells, np.sqrt((nu - 2.0) / rng_t.chisquare(nu, size=(dev.shape[0], 1))), 1.0)
    d = dev.reshape(dev.shape[0], -1, k)                         # (draws, assets, horizons)
    legs = np.diff(d, axis=2, prepend=0.0)                       # each leg's new move
    t_a = t_cells.reshape(-1, k)[:, 0]
    for j in range(k):                                           # a new chi-square per leg, shared by the leg's Student-t assets
        w = np.sqrt((nu - 2.0) / rng_t.chisquare(nu, size=(dev.shape[0], 1)))
        legs[:, :, j] *= np.where(t_a, w, 1.0)
    return np.cumsum(legs, axis=2).reshape(dev.shape[0], -1)

def m13_draws(U, origin, steps_window=WINDOW, drift_rule=True, student_t=True, ewma_halflife=None, t_per_leg=False):
    """ewma_halflife: None = plain sd of the window (M1.3), 63 = EWMA sd (M1.5).
    t_per_leg: False = one Student-t chi-square per draw for the whole path (M1.5);
               True  = a new chi-square per draw for every leg, like production (M1.6)."""
    by_asset, x, log_mode, last = fit(U, origin, steps_window)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    if ewma_halflife is not None:                       # M1.5: EWMA sd per asset; correlation from the window
        sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
        sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])
        sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    def centre(a, h):
        anchor = 0.0 if log_mode[a] else last[a]
        use_drift = U["rules"][a]["drift"] if drift_rule else True
        return anchor + (s[h] * mu[ai[a]] if use_drift else 0.0)
    mean = np.array([centre(a, h) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"]) if student_t else None
    if nu is not None and t_cells.any() and not t_per_leg:      # M1.5: one chi-square for the whole path
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    if nu is not None and t_cells.any() and t_per_leg:          # M1.6: production's design, one chi-square per leg
        dev = t_per_leg_scale(dev, cells, sorted(horizons), t_cells, nu, np.random.default_rng([seed, 7]))
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out


cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

one_h = next(u for u in UNIT_IDS if len(UNITS_DATA[u]["horizons"]) == 1 and UNITS_DATA[u]["rules"][UNITS_DATA[u]["assets"][0]]["shock"] == "student_t")
U = UNITS_DATA[one_h]
same = np.array_equal(m13_draws(U, U["asof"], ewma_halflife=63), m13_draws(U, U["asof"], ewma_halflife=63, t_per_leg=True))
print(f"check 2 — {one_h} (1 horizon): M1.5 and M1.6 draws identical: {same}"); assert same

def nu_at_asof(U):
    return fit_nu(fit(U, U["asof"], WINDOW)[1], [0])
two_h = next(u for u in UNIT_IDS if len(UNITS_DATA[u]["assets"]) == 1 and len(UNITS_DATA[u]["horizons"]) == 2
             and asset_type(UNITS_DATA[u]["assets"][0]) == "rates" and nu_at_asof(UNITS_DATA[u]) is not None)
U = UNITS_DATA[two_h]; a = U["assets"][0]
kurt = lambda v: float(np.mean((v - v.mean()) ** 4) / np.var(v) ** 2 - 3)
rows = {}
for name, kw in (("M1.5", {}), ("M1.6", {"t_per_leg": True})):
    d = m13_draws(U, U["asof"], ewma_halflife=63, **kw)[:, 0, :]
    leg1, leg2 = d[:, 0] - d[:, 0].mean(), (d[:, 1] - d[:, 0]) - (d[:, 1] - d[:, 0]).mean()
    big1, big2 = np.abs(leg1) > np.quantile(np.abs(leg1), 0.9), np.abs(leg2) > np.quantile(np.abs(leg2), 0.9)
    rows[name] = {"sd leg 1": leg1.std(), "sd leg 2": leg2.std(), "corr(leg 1, leg 2)": np.corrcoef(leg1, leg2)[0, 1],
                  "corr(|leg 1|, |leg 2|)": np.corrcoef(np.abs(leg1), np.abs(leg2))[0, 1],
                  "P(leg 2 in its top 10% | leg 1 in its top 10%)": (big1 & big2).sum() / big1.sum(),
                  "excess kurtosis, horizon 1": kurt(d[:, 0]), "excess kurtosis, horizon 2": kurt(d[:, 1])}
print(f"check 3 — {U['unit']} ({a}, horizons {U['horizons']}, nu fitted = {nu_at_asof(U):.1f}), 500 draws at the as-of:")
display(pd.DataFrame(rows).T)

VARIANTS = {"M1.5": dict(ewma_halflife=63), "M1.6": dict(ewma_halflife=63, t_per_leg=True)}
def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": o, "model": m, **parts(U["card"], m13_draws(U, o, **kw).reshape(500, -1), y)}
            for o, y in BACKTEST[u] for m, kw in VARIANTS.items()]
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // 2:,} dates x 2 models in {time.time() - t0:.0f}s")
d = float((vs_m0(G[G.model == "M1.5"]) - vs_m0(pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"))).abs().max())
print(f"check 1 — t_per_leg=False vs notebook 09's saved M1.5, competition score per unit: max |difference| = {d:.1e}")
assert d < 1e-12
M16 = G[G.model == "M1.6"].drop(columns="model").reset_index(drop=True)

check 2 — t2-F1-ai-mom-2024 (1 horizon): M1.5 and M1.6 draws identical: True
check 3 — t2-F1-considerable-period-2003 (UST_10Y, horizons [126, 189], nu fitted = 10.4), 500 draws at the as-of:


,sd leg 1,sd leg 2,"corr(leg 1, leg 2)","corr(|leg 1|, |leg 2|)",P(leg 2 in its top 10% | leg 1 in its top 10%),"excess kurtosis, horizon 1","excess kurtosis, horizon 2"
M1.5,0.7946,0.5657,0.02898,0.1712,0.32,0.2044,1.051
M1.6,0.7946,0.5562,0.02035,0.09429,0.18,0.2044,0.6003


scored 23,635 dates x 2 models in 5s
check 1 — t_per_leg=False vs notebook 09's saved M1.5, competition score per unit: max |difference| = 0.0e+00


## 2 — Every model on the same backtest

In [3]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "production (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.3 (plain sd, 260)": pd.read_parquet(DATA / "m1_3_backtest_scores.parquet"),
          "M1.4 (best nu per family)": pd.read_parquet(DATA / "m1_4_backtest_scores.parquet"),
          "M1.5 (EWMA sd, halflife 63)": pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"),
          "M1.6 (M1.5 + new t per leg)": M16}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
production (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
"M1.3 (plain sd, 260)",0.8951,0.9995,0.9194,0.9763,0.9521,81/103
M1.4 (best nu per family),0.8929,0.999,0.9191,0.9769,0.9516,81/103
"M1.5 (EWMA sd, halflife 63)",0.8932,0.99,0.9141,0.9642,0.9444,85/103
M1.6 (M1.5 + new t per leg),0.8939,0.99,0.9151,0.9642,0.9448,85/103


## 3 — M1.5 vs M1.6 (and production)

Only units with 2 or more horizons can change (a 1-horizon unit has one leg). The score-part table shows each part ÷ M0's
(geometric mean over units): **joint** is the variogram part production's design targets.

In [4]:
multi_h = [u for u in UNIT_IDS if len(UNITS_DATA[u]["horizons"]) > 1]
multi_a = [u for u in multi_h if len(UNITS_DATA[u]["assets"]) > 1]
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE),
                         "assets": pd.Series({u: len(UNITS_DATA[u]["assets"]) for u in UNIT_IDS}),
                         "horizons": pd.Series({u: len(UNITS_DATA[u]["horizons"]) for u in UNIT_IDS}),
                         "M1.5 vs M0": SC["M1.5 (EWMA sd, halflife 63)"], "M1.6 vs M0": SC["M1.6 (M1.5 + new t per leg)"],
                         "production vs M0": SC["production (text off)"]})
def summary(g):
    g2 = g[g.horizons > 1]
    return {"units": len(g), "M1.5": g["M1.5 vs M0"].mean(), "M1.6": g["M1.6 vs M0"].mean(), "production": g["production vs M0"].mean(),
            "M1.6 better than M1.5 (2+ horizon units)": f'{(g2["M1.6 vs M0"] < g2["M1.5 vs M0"]).sum()}/{len(g2)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)

PART = {}
for name in ["production (text off)", "M1.5 (EWMA sd, halflife 63)", "M1.6 (M1.5 + new t per leg)"]:
    m = models[name].groupby("unit")[cats].mean()
    for group, us in ((f"F3 ({(fam == 'T2-F3').sum()} units)", list(fam[fam == "T2-F3"].index)),
                      (f"multi-asset, 2+ horizons ({len(multi_a)})", multi_a), (f"all 2+ horizons ({len(multi_h)})", multi_h)):
        for c in cats:
            PART[(group, c)] = PART.get((group, c), {}) | {name: geo(m.loc[us, c] / mean0.loc[us, c])}
print("score parts ÷ M0 (geometric mean over units):")
display(pd.DataFrame(PART).T.rename_axis(["units", "part"]))
print("per unit, 2+ horizons:")
display(PER_UNIT.loc[multi_h].rename_axis("unit").reset_index().set_index(["family", "unit"]).sort_index())

per family:


,units,M1.5,M1.6,production,M1.6 better than M1.5 (2+ horizon units)
family,,,,,
T2-F1,23,0.8932,0.8939,0.9288,6/18
T2-F2,27,0.99,0.99,1.011,0/0
T2-F3,22,0.9141,0.9151,0.9274,8/22
T2-F4,31,0.9642,0.9642,1.003,0/0
all,103,0.9444,0.9448,0.9726,14/40


per asset type:


,units,M1.5,M1.6,production,M1.6 better than M1.5 (2+ horizon units)
asset type,,,,,
cpi,2,0.899,0.899,1.308,0/1
equity_factor,16,0.9665,0.9667,0.9935,1/3
fx,35,0.9614,0.9619,0.9819,5/11
fx+rates,5,0.933,0.9342,0.9429,1/5
payrolls,1,0.9682,0.9682,1.624,0/0
rates,43,0.9234,0.9236,0.9285,7/19
unemployment,1,1.024,1.024,1.036,0/1


score parts ÷ M0 (geometric mean over units):


production (text off)  M1.5 (EWMA sd, halflife 63)  M1.6 (M1.5 + new t per leg)
units                         part                                                                                     
F3 (22 units)                 marginal                 0.9155                       0.9062                       0.9065
                              joint                    0.9243                       0.9073                       0.9079
                              tail                     0.9453                       0.9287                        0.932
multi-asset, 2+ horizons (23) marginal                 0.9339                       0.9227                       0.9228
                              joint                      0.94                       0.9238                       0.9244
                              tail                     0.9552                       0.9303                       0.9338
all 2+ horizons (40)          marginal                 0.8908                       0.8759                       0.8764
                              joint                    0.9399                       0.9294                       0.9299
                              tail                     0.9571                       0.9096                       0.9121

per unit, 2+ horizons:


asset type  assets  horizons  M1.5 vs M0  M1.6 vs M0  production vs M0
family unit                                                                                                        
T2-F1  t2-F1-aud-on-hold-2016                        fx       1         2      0.9559      0.9598            0.9478
       t2-F1-cad-boc-2017                            fx       1         2      0.9769      0.9756             1.037
       t2-F1-chf-highly-valued-2021                  fx       1         2      0.8608      0.8675            0.8065
       t2-F1-conflicting-texts-2024               rates       1         2      0.9757      0.9772             1.026
       t2-F1-considerable-period-2003             rates       1         2      0.9192      0.9118            0.9141
       t2-F1-conundrum-2005                       rates       1         2      0.6933      0.6925            0.6256
       t2-F1-cpi-glidepath-2023                     cpi       1         2      0.8749      0.8749             1.434
       t2-F1-eur-range-2017                          fx       1         2      0.8195      0.8245            0.8291
       t2-F1-greater-confidence-2024              rates       1         2      0.9019      0.9024            0.9257
       t2-F1-hawkish-cut-2024                     rates       1         2      0.8734      0.8749            0.8902
       t2-F1-jpy-ycc-flex-2018                       fx       1         2      0.8724      0.8685            0.9337
       t2-F1-liftoff-telegraph-2015               rates       1         2      0.8888      0.8939            0.7836
       t2-F1-measured-pace-2004                   rates       1         2      0.6058      0.6075            0.5517
       t2-F1-pause-2006                           rates       2         2       1.291        1.29             1.372
       t2-F1-qe3-end-2014                         rates       1         2      0.8196      0.8192            0.8437
       t2-F1-sahm-watch-2024               unemployment       1         2       1.024       1.024             1.036
       t2-F1-third-cut-2019                       rates       1         2       1.016        1.02             1.116
       t2-F1-ust-positioning-2018                 rates       1         2      0.8352      0.8357            0.8929
T2-F3  t2-F3-bear-flattener-2022                  rates       4         2      0.9374      0.9371            0.9131
       t2-F3-boj-ust-channel-2023              fx+rates       2         2      0.9236      0.9305            0.9491
       t2-F3-brexit-joint-2016                       fx       4         2       1.002       1.009            0.9746
       t2-F3-conundrum-joint-2005                 rates       4         2      0.8016      0.8042            0.8375
       t2-F3-covid-curve-2020                     rates       4         2      0.9239      0.9212            0.9261
       t2-F3-cpi-shock-cross-2022         equity_factor       2         2      0.9737      0.9768            0.9979
       t2-F3-divergence-2014                   fx+rates       3         2      0.9238      0.9256            0.9335
       t2-F3-dollar-squeeze-2020                     fx      10         2      0.8811      0.8774            0.8867
       t2-F3-dollar-vortex-2022                fx+rates       4         2      0.9565      0.9577            0.9365
       t2-F3-dots-vs-oil-2014                     rates       3         2      0.8224      0.8217             0.833
       t2-F3-election-2024-joint               fx+rates       4         2      0.9363      0.9323            0.9338
       t2-F3-fragile-five-joint-2013                 fx       2         2      0.9846      0.9895             1.092
       t2-F3-funding-flip-2024                       fx       4         2      0.9637      0.9632            0.9737
       t2-F3-hiking-year-curve-2018               rates       4         2      0.8832      0.8834             0.919
       t2-F3-inversion-persistence-2019           rates       2         2      0.8864      0.8884    

## 4 — Save

In [5]:
M16.to_parquet(DATA / "m1_6_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_6_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m1_6_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_6_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for f in ("m1_6_backtest_scores.parquet", "m1_6_vs_m0_per_unit.csv", "m1_6_vs_m0_per_family.csv", "m1_6_vs_m0_by_asset_type.csv"):
    print(f"data/{f:<30} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/m1_6_backtest_scores.parquet      867.1 kB
data/m1_6_vs_m0_per_unit.csv            10.2 kB
data/m1_6_vs_m0_per_family.csv           0.4 kB
data/m1_6_vs_m0_by_asset_type.csv        0.6 kB


## 5 — In short

In [6]:
for name in LADDER.index:
    print(f"  {name:<30} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
print(f"M1.6 vs M1.5 on the {len(multi_h)} units with 2+ horizons: better on {FAMILY.loc['all', 'M1.6 better than M1.5 (2+ horizon units)']}")
print("per family M1.6 (M1.5): " + "; ".join(f"{f} {FAMILY.loc[f, 'M1.6']:.4f} ({FAMILY.loc[f, 'M1.5']:.4f})" for f in FAMILY.index if f != "all"))

  M0                             1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)          0.9976   (better than M0 on 25/103)
  production (text off)          0.9726   (better than M0 on 74/103)
  M1.3 (plain sd, 260)           0.9521   (better than M0 on 81/103)
  M1.4 (best nu per family)      0.9516   (better than M0 on 81/103)
  M1.5 (EWMA sd, halflife 63)    0.9444   (better than M0 on 85/103)
  M1.6 (M1.5 + new t per leg)    0.9448   (better than M0 on 85/103)
M1.6 vs M1.5 on the 40 units with 2+ horizons: better on 14/40
per family M1.6 (M1.5): T2-F1 0.8939 (0.8932); T2-F2 0.9900 (0.9900); T2-F3 0.9151 (0.9141); T2-F4 0.9642 (0.9642)


## 6 — Why a new χ² per leg does not help

Check 3 in §1 shows it on one draw set:

1. **The sd and the correlation do not change** (up to the noise of 500 draws). Leg 1 is the same numbers; leg 2's sd is
   0.566 vs 0.556, and corr(leg 1, leg 2) is about 0 in both. So the CRPS and the variogram see almost the same forecast.
   That is why the joint part moves by less than 0.1%.
2. **Big moves stop arriving together.** Under M1.5 a wild draw is wild on both legs: corr(|leg 1|, |leg 2|) = 0.17, and
   when leg 1 is in its top 10%, leg 2 is too 32% of the time. Under M1.6 those drop to 0.09 and 18% (independent legs
   would give 0 and 10%). Real markets have *volatility clustering*: a wild month is usually followed by
   another one. M1.5's shared χ² is a crude version of that; M1.6 throws it away.
3. **Horizon 2 gets thinner tails.** Horizon 2 is leg 1 + leg 2. With two independent χ², the sum of two fat-tailed legs is
   closer to normal. So the excess kurtosis at horizon 2 drops (1.05 → 0.60 in check 3; horizon 1 is unchanged at 0.20). The tail pinball score at horizon 2 then misses the big moves
   more often, and the tail part gets worse (F3: 0.929 → 0.932 × M0).

So production's joint design is already in M1.5 where it helps: the accumulating path (1) and the date-aligned cross-asset
correlation (2). The per-leg χ² (3) is the one piece that does not help on this backtest.